In [14]:
from pathlib import Path
import sys
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
repo_root

WindowsPath('C:/Users/leungk/OneDrive - EllisDon Corporation/Documents/Other/github_repos/CBG_analysis')

# Food Parsing and Units
Explode meals into item-level rows, standardize quantities/units, and convert to grams so macros can be linked to glucose responses.

In [15]:
from pathlib import Path
import pandas as pd
from src import io_excel, parse_foods, defaults, aggregate

repo_root = Path('..').resolve()
excel_path = repo_root / 'data/source_data/20251218_Trudy_Meals.xlsx'
defaults_path = repo_root / 'config/defaults_food_items.yaml'

print(f'Loading clean events from {excel_path}')
clean_events = io_excel.load_clean_events(excel_path)
print(f'Clean events: {len(clean_events)} rows')

Loading clean events from C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\data\source_data\20251218_Trudy_Meals.xlsx
Clean events: 206 rows


C:\Users\leungk\OneDrive - EllisDon Corporation\Documents\Other\github_repos\CBG_analysis\src\io_excel.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_parsed = pd.to_datetime(time_series, errors="coerce").dt.time


In [16]:
# Explode clean events into item rows
items = parse_foods.explode_food_items(clean_events)
print(f'Exploded to items: {len(items)} rows across {items.meal_id.nunique()} meals')

Exploded to items: 644 rows across 205 meals


In [17]:
# Apply defaults (qty/unit/grams)
cfg = defaults.load_defaults_config(defaults_path)
items = defaults.apply_default_rules(items, cfg)
assumed_rate = items['assumed_100g_flag'].mean() if 'assumed_100g_flag' in items else 0.0
print(f'Defaults applied; assumed_100g_flag rate: {assumed_rate:.3f}')

Defaults applied; assumed_100g_flag rate: 0.000


In [18]:
# Convert to grams and inspect distributions
items = aggregate.compute_grams(items)
print('Converted to grams; grams_final stats:')
print(items['grams_final'].describe())
print('Sample items:')
print(items[['meal_id', 'food_name_std', 'grams_final', 'assumed_100g_flag']].head(10))

Converted to grams; grams_final stats:
count    644.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: grams_final, dtype: float64
Sample items:
   meal_id               food_name_std  grams_final  assumed_100g_flag
0        1                      2 eggs        100.0               True
1        1              1 5 oz of beef        100.0               True
2        1             2 cups choy sum        100.0               True
3        2                         NaN        100.0              False
4        3                1 cup quinoa        100.0               True
5        3                      1 kiwi        100.0               True
6        3     avocado oil for cooking        100.0              False
7        3         2 cups chicken soup        100.0               True
8        3         1 5 piece pork chop        100.0               True
9        3  1 cup stir fry green beans        100.0               True


In [19]:
# Inspect assumptions/defaults applied
assumed_mask = items['assumed_100g_flag'] if 'assumed_100g_flag' in items else pd.Series(False, index=items.index)
default_mask = items['default_applied_flag'] if 'default_applied_flag' in items else pd.Series(False, index=items.index)
flagged = items[assumed_mask | default_mask]
print(f'Items with assumptions/defaults: {len(flagged)}')
print(flagged[['food_name_std', 'grams_final', 'assumed_100g_flag']].head(20))

Items with assumptions/defaults: 644
                 food_name_std  grams_final  assumed_100g_flag
0                       2 eggs        100.0               True
1               1 5 oz of beef        100.0               True
2              2 cups choy sum        100.0               True
3                          NaN        100.0              False
4                 1 cup quinoa        100.0               True
5                       1 kiwi        100.0               True
6      avocado oil for cooking        100.0              False
7          2 cups chicken soup        100.0               True
8          1 5 piece pork chop        100.0               True
9   1 cup stir fry green beans        100.0               True
10   0 5 cup blanched choy sum        100.0               True
11                         NaN        100.0              False
12                        none        100.0              False
13     avocado oil for cooking        100.0              False
14           1 pie